# CAP07-Aggregations

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Spark Guide WSL")
    .getOrCreate()
)

print(spark.version)

3.5.1


In [ ]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("C:\\GitHubProjects\\Spark-Guide\\datasets\\retail-data\\all\\*.csv")\
    .coalesce(5)

df.cache()
df.printSchema()
df.createOrReplaceTempView("dfTable")

In [ ]:
spark.sql("SELECT * FROM dfTable LIMIT 5").show()

# Aggregation Functions

### Example 1: count

In [ ]:
from pyspark.sql.functions import count

df.select(count("StockCode")).show()

### Example 2: coundDistinct

In [ ]:
from pyspark.sql.functions import countDistinct

df.select(countDistinct("StockCode")).show()

### Example 3: approx_count_distinct

In [ ]:
from pyspark.sql.functions import approx_count_distinct

df.select(approx_count_distinct("StockCode", 0.1)).show()

### Example 4: first and last

In [ ]:
from pyspark.sql.functions import first, last

df.select(first("StockCode"), last("StockCode")).show()

### Example 5: min and max

In [ ]:
from pyspark.sql.functions import min, max

df.select(min("Quantity"), max("Quantity")).show()

### Example 6: sum

In [ ]:
from pyspark.sql.functions import sum

df.select(sum("Quantity")).show()

### Example 7: sumDistinct

In [ ]:
from pyspark.sql.functions import sum_distinct

df.select(sum_distinct("Quantity")).show()

### Example 8: avg

In [ ]:
from pyspark.sql.functions import sum, count, avg, expr

df.select(
    count("Quantity").alias("total_transactions"),
    sum("Quantity").alias("total_purchases"),
    avg("Quantity").alias("avg_purchases"),
    expr("mean(Quantity)").alias("mean_purchases")) \
  .selectExpr(
      "total_purchases / total_transactions",
      "avg_purchases",
      "mean_purchases"
  ).show()

# Variance and Standard Deviation

### Example 1: var and stddev

In [ ]:
from pyspark.sql.functions import var_pop, var_samp, stddev_pop, stddev_samp

df.select(
    var_pop("Quantity").alias("var_pop"),
    var_samp("Quantity").alias("var_samp"),
    stddev_pop("Quantity").alias("stddev_pop"),
    stddev_samp("Quantity").alias("stddev_samp")
).show()

### Example 2: skewness and kurtosis

In [ ]:
from pyspark.sql.functions import skewness, kurtosis

df.select(
    skewness("Quantity").alias("skewness"),
    kurtosis("Quantity").alias("kurtosis")
).show()

### Example 3: covariance and correlation

In [ ]:
from pyspark.sql.functions import corr, covar_pop, covar_samp

df.select(corr("InvoiceNo", "Quantity"), covar_samp("InvoiceNo", "Quantity"),
    covar_pop("InvoiceNo", "Quantity")
).show()

### Example 4: Aggregating to Complex Types

In [ ]:
from pyspark.sql.functions import collect_set, collect_list

df.agg(collect_set("Country"), collect_list("Country")).show()

### Example 5: Grouping with Expressions

In [ ]:
from pyspark.sql.functions import count

df.groupBy("InvoiceNo").agg(
    count("Quantity").alias("quan"),
    expr("count(Quantity)")
).show()

### Example 6: Grouping with Maps

In [ ]:
df.groupBy("InvoiceNo").agg(expr("avg(Quantity)"), expr("stddev_pop(Quantity)")).show()

### Example 7: Window Functions

In [ ]:
from pyspark.sql.functions import col, to_date, desc, max, dense_rank, rank
from pyspark.sql.window import Window

dfWitDate = df.withColumn("date", to_date(col("InvoiceDate"), "MM/d/yyyy"))
dfWitDate.createOrReplaceTempView("dfWitDate")

windowSpec = Window \
    .partitionBy("CustomerId", "date") \
    .orderBy(desc("Quantity")) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

maxPurchaseQuantity = max(col("Quantity")).over(windowSpec)

purchaseDenseRank = dense_rank().over(windowSpec)
purchaseRank = rank().over(windowSpec)

dfWitDate.where("CustomerId IS NOT NULL").orderBy("CustomerId") \
    .select(
        col("CustomerId"),
        col("date"),
        col("Quantity"),
        purchaseRank.alias("quantityRank"),
        purchaseDenseRank.alias("quantityDenseRank"),
        maxPurchaseQuantity.alias("maxPurchaseQuantity")
    ).show()

# End